# บทเรียน 11 - โปรโตคอลบริบทของโมเดล (MCP)

The **โปรโตคอลบริบทของโมเดล (MCP)** เป็นมาตรฐานแบบเปิดที่ช่วยให้เอเยนต์สามารถค้นพบและใช้เครื่องมือ ทรัพยากร และแหล่งข้อมูลได้อย่างไดนามิกขณะรันไทม์ แทนที่จะใส่เครื่องมือแบบคงที่ลงในเอเยนต์ MCP อนุญาตให้เอเยนต์เชื่อมต่อกับเซิร์ฟเวอร์ภายนอกที่เปิดเผยความสามารถตามความต้องการ

ในบทเรียนนี้ คุณจะได้เรียนรู้:
- MCP คืออะไรและทำไมจึงสำคัญต่อระบบเอเยนต์
- สถาปัตยกรรมไคลเอนต์-เซิร์ฟเวอร์ของ MCP ทำงานอย่างไร
- วิธีสร้างเอเยนต์ที่ใช้การค้นหาเครื่องมือแบบ MCP


## การตั้งค่า

**ข้อกำหนดเบื้องต้น:**
- โครงการ Azure AI Foundry ที่มีการปรับใช้โมเดลแล้ว
- เรียกใช้ `az login` เพื่อการยืนยันตัวตน


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## โปรโตคอลบริบทของโมเดล (MCP) คืออะไร?

MCP กำหนดวิธีมาตรฐานให้กับเอเยนต์ AI ในการค้นหาและโต้ตอบกับเครื่องมือและแหล่งข้อมูลภายนอก:

- **MCP Server**: เปิดเผยเครื่องมือ, ทรัพยากร, และพรอมต์ผ่านโปรโตคอลมาตรฐาน
- **MCP Client**: ตัวรันไทม์ของเอเยนต์ที่เชื่อมต่อกับเซิร์ฟเวอร์และค้นพบความสามารถที่มีอยู่
- **Dynamic Discovery**: เอเยนต์ไม่จำเป็นต้องมีเครื่องมือแบบฮาร์ดโค้ด — พวกมันค้นพบสิ่งที่มีให้ในเวลารันไทม์

สิ่งนี้มีประสิทธิภาพสำหรับการสร้างระบบเอเยนต์ที่ขยายได้ ซึ่งสามารถเพิ่มความสามารถใหม่ๆ โดยไม่ต้องแก้ไขโค้ดของเอเยนต์


## วิธีการทำงานของ MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. เอเยนต์ (MCP client) เชื่อมต่อกับเซิร์ฟเวอร์ MCP
2. เซิร์ฟเวอร์ตอบกลับด้วยรายการของเครื่องมือที่มีและสคีมาของพวกมัน
3. เอเยนต์สามารถเรียกใช้เครื่องมือที่ค้นพบใดๆ ได้ระหว่างกระบวนการให้เหตุผลของมัน
4. ผลลัพธ์จะไหลย้อนกลับผ่านโปรโตคอลเดียวกัน


## จำลองการค้นหาเครื่องมือ MCP

เนื่องจากเซิร์ฟเวอร์ MCP ของจริงต้องการกระบวนการเซิร์ฟเวอร์ที่กำลังทำงาน เราจะสาธิตรูปแบบโดยใช้ฟังก์ชัน `@tool` ที่จำลองสิ่งที่บริการที่พักที่เชื่อมต่อกับ MCP จะให้.

ในการใช้งานจริง เครื่องมือเหล่านี้จะถูกค้นหาแบบไดนามิกจากเซิร์ฟเวอร์ MCP แทนที่จะถูกกำหนดไว้ในเครื่อง.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## สร้างเอเย่นต์ด้วยเครื่องมือสไตล์ MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ในสภาพแวดล้อมการผลิต

ในสภาพแวดล้อมการผลิต MCP เปิดใช้งานรูปแบบที่ทรงพลัง:

- **การค้นพบเครื่องมือแบบไดนามิก**: เอเยนต์เชื่อมต่อกับเซิร์ฟเวอร์ MCP และค้นพบเครื่องมือขณะรันไทม์
- **สถาปัตยกรรมที่แยกส่วน**: ผู้ให้บริการเครื่องมือสามารถอัปเดตโดยอิสระจากเอเยนต์
- **การแชร์ข้ามองค์กร**: ทีมสามารถเปิดเผยความสามารถผ่านเซิร์ฟเวอร์ MCP ที่เอเยนต์ใด ๆ ก็สามารถใช้งานได้
- **Microsoft Agent Framework support**: MAF มีการสนับสนุนไคลเอนต์ MCP ในตัวผ่านการผสานรวม `mcp`

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**เรียนรู้เพิ่มเติม:**
- [ข้อกำหนด MCP](https://modelcontextprotocol.io/)
- [การสนับสนุน MCP ของ Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## สรุป

ในบทเรียนนี้ คุณได้เรียนรู้:
- **MCP** เป็นมาตรฐานเปิดสำหรับการค้นหาเครื่องมือแบบไดนามิกระหว่างเอเย่นต์และผู้ให้บริการเครื่องมือ
- สถาปัตยกรรมแบบ **ไคลเอนต์-เซิร์ฟเวอร์** ช่วยให้เอเย่นต์ค้นพบความสามารถขณะรันไทม์
- MCP ทำให้เกิดระบบเอเย่นต์ที่ **ขยายได้และแยกส่วน** ซึ่งสามารถเพิ่มเครื่องมือได้โดยไม่ต้องเปลี่ยนแปลงโค้ด
- Microsoft Agent Framework มี **การสนับสนุน MCP ในตัว** สำหรับการใช้งานในสภาพแวดล้อมการผลิต


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารฉบับนี้ได้รับการแปลโดยบริการแปลด้วย AI Co‑op Translator (https://github.com/Azure/co-op-translator). แม้เราจะพยายามให้การแปลมีความถูกต้อง โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำได้ เอกสารต้นฉบับในภาษาต้นฉบับควรถูกถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ แนะนำให้ใช้การแปลโดยนักแปลมืออาชีพ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดที่เกิดขึ้นจากการใช้การแปลนี้.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
